In [81]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from hgrs_model import generate_zone_embedding

In [82]:
!pip install xarray netCDF4 lightgbm xgboost -q

import xarray as xr
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\jhans\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [84]:
ds = xr.open_dataset(r"C:\Users\jhans\Downloads\IMDAA_merged_1.08_1990_2020.nc")

df_weather = ds.to_dataframe().reset_index()

df_weather = df_weather.sample(frac=0.01, random_state=42)
df_weather = df_weather[['time', 'lat', 'lon', 'TMP_2m', 'APCP_sfc']]

df_weather.rename(columns={
    'TMP_2m': 'temperature',
    'APCP_sfc': 'rainfall'
}, inplace=True)

print("Weather Loaded:", df_weather.shape)

Weather Loaded: (463790, 5)


In [85]:
import os

aqi_path = r"C:\Users\jhans\Downloads\gig-protector-pro\ml-engine\data\aqi\\"

aqi_files = [f for f in os.listdir(aqi_path) if f.endswith(".csv")]
aqi_list = []

for file in aqi_files:
    temp = pd.read_csv(aqi_path + file)

    temp.columns = [col.lower().strip() for col in temp.columns]

    print(f"\nProcessing: {file}")
    print(temp.columns)

    if 'timestamp' not in temp.columns or 'pm2.5' not in temp.columns:
        print(f"Skipping {file} (missing required columns)")
        continue

    temp = temp[['timestamp', 'pm2.5']].copy()

    temp.rename(columns={
        'timestamp': 'date',
        'pm2.5': 'pm25'
    }, inplace=True)

    temp['date'] = pd.to_datetime(temp['date'], errors='coerce')
    temp.dropna(inplace=True)
    
    temp['aqi'] = temp['pm25'] * 4
    temp['aqi'] = temp['aqi'].clip(0, 500)

    city = file.split("_")[0]
    temp['city'] = city

    aqi_list.append(temp[['date', 'aqi', 'city']])

df_aqi = pd.concat(aqi_list, ignore_index=True)

print("\nAQI Dataset Ready:", df_aqi.shape)
df_aqi.head()


Processing: bengaluru_combined.csv
Index(['timestamp', 'location', 'pm2.5', 'pm10', 'no2', 'nh3', 'so2', 'co',
       'o3'],
      dtype='str')

Processing: chennai_combined.csv
Index(['timestamp', 'location', 'pm2.5', 'pm10', 'no2', 'nh3', 'so2', 'co',
       'o3'],
      dtype='str')

Processing: delhi_combined.csv
Index(['timestamp', 'location', 'pm2.5', 'pm10', 'no2', 'nh3', 'so2', 'co',
       'o3'],
      dtype='str')

Processing: gwalior_combined.csv
Index(['timestamp', 'location', 'pm2.5', 'pm10', 'no2', 'nh3', 'so2', 'co',
       'o3'],
      dtype='str')

Processing: hyderabad_combined.csv
Index(['timestamp', 'location', 'pm2.5', 'pm10', 'no2', 'nh3', 'so2', 'co',
       'o3'],
      dtype='str')

Processing: jaipur_combined.csv
Index(['timestamp', 'location', 'pm2.5', 'pm10', 'no2', 'nh3', 'so2', 'co',
       'o3'],
      dtype='str')

Processing: kolkata_combined.csv
Index(['timestamp', 'location', 'pm2.5', 'pm10', 'no2', 'nh3', 'so2', 'co',
       'o3'],
      dtype='str'

,date,aqi,city
0,2020-02-01,174.68,bengaluru
1,2020-03-01,122.32,bengaluru
2,2020-04-01,265.40,bengaluru
3,2020-05-01,192.00,bengaluru
4,2020-06-01,95.00,bengaluru


In [86]:
df_delivery = pd.read_csv(
    r"C:\Users\jhans\Downloads\gig-protector-pro\ml-engine\data\deliverytime\deliverytime.csv"
)


df_delivery.columns = [col.lower().strip() for col in df_delivery.columns]
df_delivery.rename(columns={
    'time_taken(min)': 'delivery_time'
}, inplace=True)

df_delivery['delivery_person_age'] = pd.to_numeric(df_delivery['delivery_person_age'], errors='coerce')
df_delivery['delivery_person_ratings'] = pd.to_numeric(df_delivery['delivery_person_ratings'], errors='coerce')

df_delivery.dropna(inplace=True)

print("Delivery Cleaned:", df_delivery.shape)

Delivery Cleaned: (45593, 11)


In [87]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

df_delivery['distance'] = haversine(
    df_delivery['restaurant_latitude'],
    df_delivery['restaurant_longitude'],
    df_delivery['delivery_location_latitude'],
    df_delivery['delivery_location_longitude']
)

from sklearn.preprocessing import LabelEncoder

le_order = LabelEncoder()
le_vehicle = LabelEncoder()

df_delivery['type_of_order'] = le_order.fit_transform(df_delivery['type_of_order'])
df_delivery['type_of_vehicle'] = le_vehicle.fit_transform(df_delivery['type_of_vehicle'])

df_delivery = df_delivery[[
    'delivery_person_age',
    'delivery_person_ratings',
    'distance',
    'type_of_order',
    'type_of_vehicle',
    'delivery_time'
]]

In [89]:
df_weather['date'] = pd.to_datetime(df_weather['time']).dt.date
df_aqi['date'] = pd.to_datetime(df_aqi['date']).dt.date

In [90]:
df_weather['zone'] = (
    df_weather['lat'].round(1).astype(str) + "_" +
    df_weather['lon'].round(1).astype(str)
)

df = pd.merge(df_weather, df_aqi, on='date', how='inner')

print("Merged Weather + AQI:", df.shape)

Merged Weather + AQI: (56535, 9)


In [91]:
min_size = min(len(df), len(df_delivery))

df = df.sample(min_size, random_state=42).reset_index(drop=True)
df_delivery = df_delivery.sample(min_size, random_state=42).reset_index(drop=True)

# Merge 
df = pd.concat([df, df_delivery], axis=1)

In [92]:
df['temp_rain'] = df['temperature'] * df['rainfall']

df['hour'] = pd.to_datetime(df['time']).dt.hour
df['month'] = pd.to_datetime(df['time']).dt.month
df['aqi_log'] = np.log1p(df['aqi'])

df['high_temp'] = (df['temperature'] > 310).astype(int)
df['heavy_rain'] = (df['rainfall'] > 10).astype(int)
df['bad_aqi'] = (df['aqi'] > 300).astype(int)

In [93]:
# =============================
# 🌦️ CREATE WEATHER CATEGORY
# =============================

def get_weather(row):
    if row['rainfall'] > 10:
        return "Rainy"
    elif row['temperature'] > 310:
        return "Hot"
    else:
        return "Normal"

df['weather'] = df.apply(get_weather, axis=1)

In [94]:
from sklearn.preprocessing import LabelEncoder

le_weather = LabelEncoder()
df['weather_enc'] = le_weather.fit_transform(df['weather'])

print("Weather encoding done:", df[['weather', 'weather_enc']].head())

Weather encoding done:   weather  weather_enc
0  Normal            1
1  Normal            1
2  Normal            1
3  Normal            1
4  Normal            1


In [95]:
df['zone'] = df['city']

df['embedding'] = [
    generate_zone_embedding(z, w)
    for z, w in zip(df['zone'], df['weather_enc'])
]

embedding_df = pd.DataFrame(
    df['embedding'].tolist(),
    columns=[f'emb_{i}' for i in range(8)]
)

df = pd.concat([df.reset_index(drop=True), embedding_df.reset_index(drop=True)], axis=1)

print("✅ Embeddings added:", embedding_df.shape)
print(df.columns)

✅ Embeddings added: (45593, 8)
Index(['time', 'lat', 'lon', 'temperature', 'rainfall', 'date', 'zone', 'aqi',
       'city', 'delivery_person_age', 'delivery_person_ratings', 'distance',
       'type_of_order', 'type_of_vehicle', 'delivery_time', 'temp_rain',
       'hour', 'month', 'aqi_log', 'high_temp', 'heavy_rain', 'bad_aqi',
       'weather', 'weather_enc', 'embedding', 'emb_0', 'emb_1', 'emb_2',
       'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7'],
      dtype='str')


In [96]:
df['delay'] = (df['delivery_time'] > df['delivery_time'].median()).astype(int)

In [97]:
X = df[[
    'temperature',
    'rainfall',
    'aqi_log',
    'delivery_person_age',
    'delivery_person_ratings',
    'distance',
    'type_of_order',
    'type_of_vehicle',
    'hour',
    'month',
    'high_temp',
    'heavy_rain',
    'bad_aqi'
] + [f'emb_{i}' for i in range(8)]]

# Regression
y_reg = df['delivery_time']

# Classification
y_clf = df['delay']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

In [80]:
print("TRAIN FEATURES:", X.shape)
print("COLUMNS:", X.columns.tolist())

TRAIN FEATURES: (45593, 21)
COLUMNS: ['temperature', 'rainfall', 'aqi_log', 'delivery_person_age', 'delivery_person_ratings', 'distance', 'type_of_order', 'type_of_vehicle', 'hour', 'month', 'high_temp', 'heavy_rain', 'bad_aqi', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7']


In [98]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

xgb = XGBRegressor(n_estimators=200)
rf = RandomForestRegressor(n_estimators=200)
lgbm = LGBMRegressor(n_estimators=200)

xgb.fit(X_train, y_train_reg)
rf.fit(X_train, y_train_reg)
lgbm.fit(X_train, y_train_reg)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007575 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1351
[LightGBM] [Info] Number of data points in the train set: 36474, number of used features: 21
[LightGBM] [Info] Start training from score 26.296184


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [99]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

def eval_reg(model, name):
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test_reg, preds))
    r2 = r2_score(y_test_reg, preds)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")

eval_reg(xgb, "XGBoost")
eval_reg(rf, "Random Forest")
eval_reg(lgbm, "LightGBM")


XGBoost
RMSE: 7.4761
R2 Score: 0.3582

Random Forest
RMSE: 7.3037
R2 Score: 0.3875

LightGBM
RMSE: 7.1717
R2 Score: 0.4094


In [100]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

xgb_clf = XGBClassifier(n_estimators=200)
rf_clf = RandomForestClassifier(n_estimators=200)
lgbm_clf = LGBMClassifier(n_estimators=200)

xgb_clf.fit(X_train, y_train_clf)
rf_clf.fit(X_train, y_train_clf)
lgbm_clf.fit(X_train, y_train_clf)

[LightGBM] [Info] Number of positive: 16578, number of negative: 19896
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007947 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1351
[LightGBM] [Info] Number of data points in the train set: 36474, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.454516 -> initscore=-0.182442
[LightGBM] [Info] Start training from score -0.182442


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [101]:
from sklearn.metrics import f1_score, roc_auc_score

def eval_clf(model, name):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:,1]

    print(f"\n{name}")
    print("F1 Score:", f1_score(y_test_clf, preds))
    print("AUC Score:", roc_auc_score(y_test_clf, probs))

eval_clf(xgb_clf, "XGBoost")
eval_clf(rf_clf, "Random Forest")
eval_clf(lgbm_clf, "LightGBM")


XGBoost
F1 Score: 0.6623761101814906
AUC Score: 0.7827660144993337

Random Forest
F1 Score: 0.6600239139099243
AUC Score: 0.7934510220625499

LightGBM
F1 Score: 0.667291722475221
AUC Score: 0.8021363272649096


In [102]:
import joblib

# Regression
joblib.dump(xgb, "xgb_reg.pkl")
joblib.dump(rf, "rf_reg.pkl")
joblib.dump(lgbm, "lgbm_reg.pkl")

# Classification
joblib.dump(xgb_clf, "xgb_clf.pkl")
joblib.dump(rf_clf, "rf_clf.pkl")
joblib.dump(lgbm_clf, "lgbm_clf.pkl")

['lgbm_clf.pkl']

In [103]:
print(X.corrwith(y_reg).sort_values(ascending=False))

delivery_person_age        0.292708
emb_3                      0.009534
emb_4                      0.008760
heavy_rain                 0.008353
temperature                0.007039
rainfall                   0.003894
type_of_order              0.002847
high_temp                  0.002585
emb_7                      0.001132
hour                       0.000499
emb_5                      0.000476
month                     -0.002473
distance                  -0.002508
emb_2                     -0.002775
emb_1                     -0.003631
bad_aqi                   -0.005525
aqi_log                   -0.009523
emb_0                     -0.010336
emb_6                     -0.010781
type_of_vehicle           -0.080572
delivery_person_ratings   -0.331103
dtype: float64


In [104]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

In [105]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [106]:
from sklearn.neural_network import MLPRegressor, MLPClassifier

In [107]:
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=200,
    random_state=42
)

mlp_reg.fit(X_train_scaled, y_train_reg)

C:\Users\jhans\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [109]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

preds_reg = mlp_reg.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test_reg, preds_reg))
r2 = r2_score(y_test_reg, preds_reg)

print("\nMLP REGRESSION")
print("RMSE:", round(rmse, 4))
print("R2:", round(r2, 4))


MLP REGRESSION
RMSE: 8.0375
R2: 0.2582


In [108]:
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=200,
    random_state=42
)

mlp_clf.fit(X_train_scaled, y_train_clf)

C:\Users\jhans\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


In [110]:
from sklearn.metrics import f1_score, roc_auc_score

preds_clf = mlp_clf.predict(X_test_scaled)
probs_clf = mlp_clf.predict_proba(X_test_scaled)[:, 1]

f1 = f1_score(y_test_clf, preds_clf)
auc = roc_auc_score(y_test_clf, probs_clf)

print("\nMLP CLASSIFICATION")
print("F1 Score:", round(f1, 4))
print("AUC Score:", round(auc, 4))


MLP CLASSIFICATION
F1 Score: 0.6243
AUC Score: 0.7501


In [111]:
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(mlp_reg, "../models/mlp_reg.pkl")
joblib.dump(mlp_clf, "../models/mlp_clf.pkl")

['../models/mlp_clf.pkl']